# <center> The Laplacian </center>

The surface Laplacian or the Current Source Density (CSD) are mathematical algorithms that transform the scalp recorded EEG into estimates of scalp level radial current flow. It acts as a spatial filter that filters out the low-frequency topographical features and, thereby, increases topographical selectivity. The application of surface Laplacian is based on the assumption that the low frequency or "broad" topographical features reflect the effect of "**volume conduction**". The surface Laplacian need is only applied to EEG data and the EEG data should have at 64 electrodes. 

Note:
**Volume Conduction**: the tangential spread of electrical fields at the boundary between the skull and the scalp. It effectively implies that the activity at a given neural generator or "**source**" can be picked up by many if not all scalp electrodes. 
 

## A very simple 1D simulation

<img align="left" src="MASCO-EEGFigures/Midline-electrodes.png" width ="150" height=150/>

To get a basic idea of what the Laplacian does when we apply it to our EEG data and why it might be useful in EEG, we will simulate some one-dimensional EEG data. An example of one-dimensional data in EEG are measures of activity from midline electrodes, for example, Fz, Cz and Pz.



In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

electrode_spacing = .1      #Define the spacing between electrodes.
space = np.arange(0, 10, electrode_spacing)
comp1 = 2 * stats.norm.pdf(space, loc=4, scale=1)     #Simulate EEG component1 (positive)
comp2 = -.25 * stats.norm.pdf(space, loc=5, scale=.5) #Simulate EEG component2 (negative)

def do_label():
    plt.legend(loc='upper right')
    plt.xlabel('Location')
    plt.ylabel('')
    
fig_ground_truth = plt.figure()
for comp, label in zip([comp1, comp2], ['Positive', 'Negative']):
    plt.plot(space, comp, label=label)
do_label()

In [ ]:
# However, in reality the two components above are superimposed. So that the little, negative one is hidden 
# by the larger positive one. The presence of the smaller negative component is suggested by the slight deformity
# of the signal between locations 6 and 7.

eeg_signal = comp1 + comp2
plt.plot(space, eeg_signal, label='EEG Signal')
do_label()

In [ ]:
# Now we calculate the first-order spatial derivative. 
# This is very simply the difference between each EEG value from one electrode to the next.

first_derivative = np.diff(eeg_signal)
space_d1 = space[:-1] + .5 * electrode_spacing # Keep locations lined up with differenced data
plt.plot(space_d1, first_derivative, label='First spatial derivative')
do_label()

In [ ]:
# Now we are going to calculate the second-order spatial derivative; the derivative of the derivative.

second_derivative = np.diff(first_derivative)
space_d2 = space[:-2] + .5 * electrode_spacing   # We want our 2nd order derivatives to coincide with each electrode.
plt.plot(space_d2, second_derivative, label='Second spatial derivative')
do_label()

In [ ]:
# The polarity is inversed, so let's correct that by multiplying by -1.

plt.plot(space_d2, -1*second_derivative, label='-1 × Second spatial derivative')
do_label()

#...and compare with our two original components...

fig_ground_truth = plt.figure()
for comp, label in zip([comp1, comp2], ['Positive', 'Negative']):
    plt.plot(space, comp, label=label)
do_label()

<div class="alert-warning">

<span style="color:black">
So comparing the original simulated components and the second-order derivative of our superimposed components,
- Can we say that we have fully recovered the original components?

- What have we managed to recover by applying the 2nd order derivative?

- What has changed?

- So what can we say about what happens when we apply the Laplacian?

</span>



</div>

## A very simple 2D simulation

Now we take one step further and take into account the fact that, in EEG, electrodes are positioned in 3D space on the head. We will, however, simplify by presuming that the scalp is flat, which allows us to look carry out a two-dimensional Laplacian. 

<img align="center" src="MASCO-EEGFigures/Topographies-2D-3D.png" width ="450" height=400/>

In [ ]:
X = Y = np.arange(-10, 10.01, .1)
XY = np.array([(x, y) for x in X for y in Y])

def show(X):
    '''Plot an image from a 2D array'''
    mx = np.abs(X).max()
    im = plt.imshow(X, cmap='seismic', vmin=-mx, vmax=mx)
    plt.colorbar(im)

In [ ]:
locs = [(0, -4), (0, 2)]  # Center of each component
peaks = [-1, 6]           # Peak amplitudes
scales = [2, 8]           # Standard deviations
V = np.zeros(XY.shape[0]) # Voltages

for loc, peak, scale in zip(locs, peaks, scales):
    cov = np.eye(2) * scale
    thisV = stats.multivariate_normal.pdf(XY, mean=loc, cov=cov)
    gain = peak / thisV.max()
    V += thisV * gain
wV = V.reshape(len(X), len(X)).T # Voltage as a 2D grid
show(wV)

In [ ]:
# First of all, lets calculate the first derivatives. 

def combine(dx, dy):

    n = dx.shape[0]
    assert(dy.shape[1]==n)
    return dx[:n, :n] + dy[:n, :n]

# Calculate the partial derivatives (derivative of voltage for x and y axis)
dv_dx = np.diff(wV, axis=0)[:-1, :] # Drop the last value on the appropriate axis to keep things symmetrical
dv_dy = np.diff(wV, axis=1)[:, :-1]
gradient = combine(dv_dx, dv_dy)
show(gradient)

In [ ]:
# Now we are going to calculate the negative second-derivatives

ddv_ddx = np.diff(dv_dx, axis=0)[:-1, :]
ddv_ddy = np.diff(dv_dy, axis=1)[:, :-1]
laplacian = -combine(ddv_ddx, ddv_ddy)  # Combine the partial derivatives. 
show(laplacian)

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

file2read = 'MASCO-EEGDATA/sub-002_ses01_task-ThreeStimAuditoryOddball_raw.fif'   # Define the path to dataset and the dataset title. 

rawIn = mne.io.read_raw_fif(file2read, preload=True) # The rawIn variable is our raw object.

In [ ]:
# Extract information from the imported raw dataset.
# Rename the channels. 

sfreq = rawIn.info['sfreq']
print("The sampling frequency is: "+str(sfreq) + "Hz")   #Print sampling frequency to screen. 
channoms = rawIn.info['ch_names']
print(channoms)                     #Print the electrode labels to screen.
hpfilt = rawIn.info['highpass']
print("High-pass filter cutoff: "+str(hpfilt)+"Hz")      #Print high-frequency cut-off frequency to screen.
lowfilt = rawIn.info['lowpass']
print("Low-pass filter cutoff: "+str(lowfilt)+"Hz")      #Print low-frequency cut-off frequency to screen. 

In [ ]:
%matplotlib qt

# Can also use %matplotlib notebook if qt doesn't work.
# We can visualise the EEG continuous data. 
# You should see a signal or channel for each electrode.
# In the following example, the window presents 10seconds of data and 10 channels.
# Set the "remove_dc" parameter to False, what has happened and why?

mne.viz.plot_raw(rawIn, events=None, duration=5.0, n_channels=10,title='Raw EEG Data', remove_dc=True)  # Presenting 10seconds of data in and 10 channels in each window.

In [ ]:
#%% RE-REFERENCE THE RAW DATA BY APPLYING A REFERENCE
# Here we do not define reference channels so MNE will apply the average of all the scalp electrodes.

rawIn.set_eeg_reference()


# Assign the standard 1020 montage to the raw object.

montage = mne.channels.make_standard_montage('standard_1020')
mne.viz.plot_montage(mne.channels.make_standard_montage('standard_1020'))

rawIn.set_montage(montage,raise_if_subset=False)  # Assign the standard 1020 montage to the raw object.

#%% Apply a FIR bandpass filter which will conserve activity in the 0.4Hz to 80Hz frequency band and reject all others. 
# As filtering transforms the raw object, we make a copy of it and apply the filter to this copy; rawfilt is now 
# our filtered object. 

picks = mne.pick_types(rawIn.info, meg=False, eeg=True, eog=False,
                       stim=False, exclude='bads')  #We only want to choose scalp electrodes

rawfilt = rawIn.copy().filter(0.5, 80,picks=picks, n_jobs=2, fir_design='firwin')


In [ ]:
#%%Plotting the topography of the average EEG activity from 600secs to 650secs
# The plot_topomap method extracts the electrode coordinate data from the 'info' data structure. 

dataint, times_int = rawfilt[picks, int(sfreq * 600):int(sfreq * 650)]
data_avg = np.mean(dataint, axis=1)

fig, ax = plt.subplots()
mne.viz.plot_topomap(data_avg, pos=rawfilt.info, axes=ax, show=False)
mne.viz.tight_layout()

plt.show()

In [ ]:

def surface_laplacian(dataIn, leg_order, m, smoothing):
    """
    This function attempts to compute the surface laplacian transform to an mne Epochs object. The 
    algorithm follows the formulations of Perrin et al. (1989) and it consists for the most part in a 
    nearly-literal translation of Mike X Cohen's 'Analyzing neural time series data' corresponding MATLAB 
    code (2014).
    
    INPUTS are:
        - epochs: mne Epochs object
        - leg_order: maximum order of the Legendre polynomial
        - m: smothness parameter for G and H
        - smoothing: smothness parameter for the diagonal of G
        - montage: montage to reconstruct the transformed Epochs object (same as in raw data import)
        
    OUTPUTS are:
        - before: unaffected reconstruction of the original Epochs object
        - after: surface laplacian transform of the original Epochs object
        
    References:
        - Perrin, F., Pernier, J., Bertrand, O. & Echallier, J.F. (1989). Spherical splines for scalp 
          potential and current density mapping. Electroencephalography and clinical Neurophysiology, 72, 
          184-187.
        - Cohen, M.X. (2014). Surface Laplacian In Analyzing neural time series data: theory and practice 
          (pp. 275-290). London, England: The MIT Press.
    """
    # import libraries
    import numpy as np
    from scipy import special
    import math
    import mne
    
    # get electrodes positions
    locs = dataIn._get_channel_positions()

    x = locs[:,0]
    y = locs[:,1]
    z = locs[:,2]

    # arrange data
    data = dataIn.get_data() # data
    orig_data_size = np.squeeze(data.shape)
    print('orig_data_size')

    numelectrodes = len(x)
    
    # normalize cartesian coordenates to sphere unit
    def cart2sph(x, y, z):
        hxy = np.hypot(x, y)
        r = np.hypot(hxy, z)
        el = np.arctan2(z, hxy)
        az = np.arctan2(y, x)
        return az, el, r

    junk1, junk2, spherical_radii = cart2sph(x,y,z)
    maxrad = np.max(spherical_radii)
    x = x/maxrad
    y = y/maxrad
    z = z/maxrad
    
    # compute cousine distance between all pairs of electrodes
    cosdist = np.zeros((numelectrodes, numelectrodes))
    for i in range(numelectrodes):
        for j in range(i+1,numelectrodes):
            cosdist[i,j] = 1 - (((x[i] - x[j])**2 + (y[i] - y[j])**2 + (z[i] - z[j])**2)/2)

    cosdist = cosdist + cosdist.T + np.identity(numelectrodes)

    # get legendre polynomials
    legpoly = np.zeros((leg_order, numelectrodes, numelectrodes))
    for ni in range(leg_order):
        for i in range(numelectrodes):
            for j in range(i+1, numelectrodes):
                #temp = special.lpn(8,cosdist[0,1])[0][8]
                legpoly[ni,i,j] = special.lpn(ni+1,cosdist[i,j])[0][ni+1]

    legpoly = legpoly + np.transpose(legpoly,(0,2,1))

    for i in range(leg_order):
        legpoly[i,:,:] = legpoly[i,:,:] + np.identity(numelectrodes)

    # compute G and H matrixes
    twoN1 = np.multiply(2, range(1, leg_order+1))+1
    gdenom = np.power(np.multiply(range(1, leg_order+1), range(2, leg_order+2)), m, dtype=float)
    hdenom = np.power(np.multiply(range(1, leg_order+1), range(2, leg_order+2)), m-1, dtype=float)

    G = np.zeros((numelectrodes, numelectrodes))
    H = np.zeros((numelectrodes, numelectrodes))
    print('Computed G and H matrices')

    for i in range(numelectrodes):
        for j in range(i, numelectrodes):
            
            g = 0
            h = 0

            for ni in range(leg_order):
                g = g + (twoN1[ni] * legpoly[ni,i,j]) / gdenom[ni]
                h = h - (twoN1[ni] * legpoly[ni,i,j]) / hdenom[ni]

            G[i,j] = g / (4*math.pi)
            H[i,j] = -h / (4*math.pi)

    G = G + G.T
    H = H + H.T

    G = G - np.identity(numelectrodes) * G[1,1] / 2
    H = H - np.identity(numelectrodes) * H[1,1] / 2

    if np.any(orig_data_size==1):
        data = data[:]
        print('No need to change dimensions')
    else:
        data = np.reshape(data, (orig_data_size[0], np.prod(orig_data_size[1:3])))

    # compute C matrix
    Gs = G + np.identity(numelectrodes) * smoothing
    GsinvS = np.sum(np.linalg.inv(Gs), 0)
    dataGs = np.dot(data.T, np.linalg.inv(Gs))
    C = dataGs - np.dot(np.atleast_2d(np.sum(dataGs, 1)/np.sum(GsinvS)).T, np.atleast_2d(GsinvS))

    # apply transform

    original = data
    surf_lap = np.transpose(np.dot(C,np.transpose(H)))
    
    ch_names = dataIn.info['ch_names']
    sfreq = dataIn.info['sfreq']
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    
    
    before = mne.io.RawArray(data=original, info=info)
    after = mne.io.RawArray(data=surf_lap, info=info)
    print('All Done')
    
    return before, after

In [ ]:
#rawfilt.drop_channels(['VEOG','EKG'])

before, after = surface_laplacian(rawfilt, 12, 3, 1e-5)

In [ ]:
# Assign the standard 1020 montage to the raw object.
after.set_montage(montage,raise_if_subset=False)  # Assign the standard 1020 montage to the raw object.

In [ ]:
dataint, times_int = rawfilt[:, int(sfreq * 600):int(sfreq * 605)]
data_avg = np.mean(dataint, axis=1)

fig, ax = plt.subplots()
mne.viz.plot_topomap(data_avg, pos=rawfilt.info, axes=ax, show=False)
mne.viz.tight_layout()

plt.show()

#After
dataint_lap, times_int = after[:, int(sfreq * 600):int(sfreq * 605)]
data_avg_lap = np.mean(dataint_lap, axis=1)

fig, ax = plt.subplots()
mne.viz.plot_topomap(data_avg_lap, pos=rawfilt.info, axes=ax, show=False)
mne.viz.tight_layout()

plt.show()